DH演習2026 · 教材 第54版 · 2026-r54 · 2026-11-26

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第54版
教材ID: 2026-r54
教材日付: 2026-11-26
活動: place_mapping
環境: Colab / Jupyter
ソース: 9dbb0fdc0476256d55eb9592df1f749ab94cdee9
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第9回：表の行を地図の点にする

確認問題まで上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

確認問題の予測は、実行する前に自分で書きます。エラーが出たら、AIに説明を求められます。確認問題の後の追加課題は授業外・任意です。

資料：第8回の難波橋の判断表から作った2行、背景地図は[OpenStreetMap](https://www.openstreetmap.org/copyright)

## 1. 表を読む

N-currentはWikidataにある現在の代表点、N-earlierは1915年に移設される前の橋で、位置を保留した行です。次の操作セルは、地図に描くための7列だけにした第9回の簡略版の表を、locations-example.csvとして保存します。第8回の作業ZIPのlocations.csvとは列が違うので、marimo版には読み込めません。同じフォルダに自分のlocations.csvがあっても、上書きしません。

その下のセルで読み込み、二行の座標を見比べてください。表の空欄は、pandasでは`NaN`（値がないことを示す表示）になります。

In [ ]:
#@title 操作：第9回の簡略版の表をlocations-example.csvとして保存する
table = """record_id,label,period_label,latitude,longitude,location_status,source_url
N-current,難波橋,現代の代表点,34.692694,135.507139,modern,https://www.wikidata.org/w/index.php?title=Q11659546&oldid=2191803708
N-earlier,難波橋,1915年の移設以前,,,unresolved,https://www.city.osaka.lg.jp/kensetsu/page/0000023795.html
"""
with open("locations-example.csv", "w", encoding="utf-8") as file:
    file.write(table)

In [ ]:
import pandas as pd

rows = pd.read_csv("locations-example.csv")
rows

## 2. 座標のある行を選ぶ

空欄を0にすると、欠けた位置を別の場所に描いてしまいます。`dropna`は緯度・経度が`NaN`の行を除き、元の`rows`は変えません。

上の表で緯度・経度が両方記入された行は何行ありますか。次のセルを実行し、最後に表示される`located`の行と照合してください。

In [ ]:
located = rows.dropna(subset=["latitude", "longitude"])
print("地図に置く行：", len(located), "位置を保留する行：", len(rows) - len(located))
located

## 3. 地図に点を置く

`located.to_dict("records")`は、表の各行を、列名をキーとする辞書にしたリストです。`folium.CircleMarker`には緯度・経度の順で渡し、`add_to(m)`で地図`m`に加えます。`radius`は円の大きさ（ピクセル）、`color`は色で、`"red"`や`"#a35a25"`のような色の名前やコードを使えます。`zoom`・`radius`・`color`を変えても、場所の判断は変わりません。

点をクリックすると、元の行の表記と時期が表示されます。`m.save("map.html")`は、地図をHTMLファイルとして保存します。左のファイル一覧から、map.htmlをダウンロードできます。

In [ ]:
import folium

zoom = 16
radius = 8
color = "darkgreen"

# 地図の中心：難波橋付近の緯度・経度
m = folium.Map(location=[34.6927, 135.5071], zoom_start=zoom)
for row in located.to_dict("records"):
    point = [row["latitude"], row["longitude"]]
    label = row["label"] + "／" + row["period_label"]
    marker = folium.CircleMarker(point, radius=radius, color=color, fill=True, popup=label)
    marker.add_to(m)
m.save("map.html")
m

地図にないN-earlierの行は、表には残っています。1915年より前の橋を地図に置くには、どの資料から何を読み取る必要があるでしょうか。

## 確認問題

練習用の表`trial_rows`は、上の二行に、緯度だけが分かる行（`practice`）を加えたものです。次の操作セルを実行し、各行の緯度と経度を見てください。

この表に2節と同じ`dropna(subset=["latitude", "longitude"])`を使うと、どの行が`folium.CircleMarker`に渡されるでしょうか。実行する前に予測し、地図に描かれる記録IDを`plotted`に、表にだけ残る記録IDを`table_only`に書いてください。

In [ ]:
#@title 操作：緯度だけがある練習用の行を加えて表示する
# 下の答えのセルで書き換える、記録IDのリスト
plotted: list[str] = []
table_only: list[str] = []
trial_rows = pd.concat([rows, pd.DataFrame([{"record_id": "practice", "label": "練習用", "period_label": "練習用", "latitude": 34.69}])], ignore_index=True)
trial_rows[["record_id", "latitude", "longitude"]]

In [ ]:
plotted = []     # 地図に描かれる記録IDのリスト。例：["A1", "A2"]
table_only = []  # 表にだけ残る記録IDのリスト

In [ ]:
#@title 答えを確かめる
all_ids = list(trial_rows["record_id"])


def read_ids(name):
    value = globals().get(name)
    if isinstance(value, str):
        print(f"ヒント：{name}は、[\"A1\"]のように記録IDを角かっこ[ ]で囲んだリストにしてください。")
        return None
    if not isinstance(value, (list, tuple)):
        print(f"ヒント：{name}には、記録IDのリストを入れてください（例：[\"A1\", \"A2\"]）。")
        return None
    ids = []
    for item in value:
        if not isinstance(item, str):
            print(f"ヒント：{name}の{item!r}は文字列ではありません。記録IDは\"N-current\"のように引用符で囲みます。")
            return None
        if item.strip() not in all_ids:
            print(f"ヒント：{item}は練習用の表にない記録IDです。表のrecord_idの列から写してください。")
            return None
        ids.append(item.strip())
    return ids


predicted = read_ids("plotted")
remaining = read_ids("table_only")
if predicted is None or remaining is None:
    print("結果：不合格（ヒントに従って直し、もう一度実行してください）")
elif not predicted and not remaining:
    print("未回答：plottedとtable_onlyに記録IDを書いてください。")
elif set(predicted) & set(remaining):
    print("結果：不合格（一つの行は、地図に描かれるか、表にだけ残るかのどちらかです）")
elif sorted(predicted + remaining) != sorted(all_ids):
    print(f"結果：不合格（練習用の表は{len(all_ids)}行です。どの記録IDも、どちらか一方に一回だけ入れてください）")
elif "practice" in predicted:
    print("結果：不合格（practiceは経度が空欄です。subsetの列のうち一つでも空欄なら、その行は外れます）")
elif "N-earlier" in predicted:
    print("結果：不合格（N-earlierは緯度も経度も空欄です）")
elif "N-current" not in predicted:
    print("結果：不合格（N-currentは緯度と経度がそろった行です）")
else:
    print("結果：合格")

次のセルを実行して、予測と照らし合わせてください。地図に渡す表から外れた行も、`trial_rows`には残っています。

In [ ]:
trial_located = trial_rows.dropna(subset=["latitude", "longitude"])
print("練習用の全行：", list(trial_rows["record_id"]))
print("地図に描く行：", list(trial_located["record_id"]))

## 授業外・任意：1908年の地図で読んだ点を加える

marimo版の2節では、1908年の地図で読んだ点を、N-earlierとは別の行に記録します。次のセルで、`N-1908`の行の`None`を、そこで読んだ緯度と経度に書き換えて実行してください。`None`は値がないことを表し、pandasの表では`NaN`になります。`None`のままなら地図の点は一つ、座標を入れると二つです。

In [ ]:
# N-1908の行のNoneを、marimo版で読んだ緯度・経度に書き換えます（例：34.69）。
two_rows = [
    {"record_id": "N-current", "label": "難波橋", "period_label": "現代の代表点", "latitude": 34.692694, "longitude": 135.507139},
    {"record_id": "N-1908", "label": "難波橋", "period_label": "1908年測図", "latitude": None, "longitude": None},
]
two_table = pd.DataFrame(two_rows)
two_located = two_table.dropna(subset=["latitude", "longitude"])
print("二行の表で地図に置く行：", len(two_located))

m2 = folium.Map(location=[34.6927, 135.5071], zoom_start=17)
for row in two_located.to_dict("records"):
    point = [row["latitude"], row["longitude"]]
    label = row["label"] + "／" + row["period_label"]
    marker = folium.CircleMarker(point, radius=8, color="darkgreen", fill=True, popup=label)
    marker.add_to(m2)
m2